In [122]:
import pandas as pd
import random

random.seed(42)

In [123]:
df = pd.read_csv('rubik_solves.csv')

df = df.replace(['n/a', 'None', 'N/A', 'null'], pd.NA)

df_cleaned = df.dropna()

print(f"Удалено строк: {len(df) - len(df_cleaned)}")
print(f"Осталось строк: {len(df_cleaned)}")
df_cleaned.head()

Удалено строк: 291
Осталось строк: 709


,solve_id,solver,solve_time,pll,pll_time,Split,STM,STPS,ETM,ETPS
4,12922,Radosław Marcinek,5.18,R' U' R U D' R2 U R' U R U' R U' R2' U D,1.31,25.3%,16,12.21,16,12.21
5,12921,Dylan Miller,4.40,U U' R2 U R' U R' U' R U' R2 D U' R' U R D' U2',1.37,31.1%,18,13.14,18,13.14
6,12920,Dylan Miller,4.60,U' R' U2 R' D' R U' R' D R U R U' R' U' R,1.03,22.4%,16,15.53,16,15.53
7,12919,Dylan Miller,5.26,U' R2 U R' U R' U' R U' R2 U' D R' U R D' U,1.53,29.1%,17,11.11,17,11.11
8,12918,Dylan Miller,6.72,U U' R' U' R U D' R2 U R' U R U' R U' R2' U' D U',1.95,29.0%,19,9.74,19,9.74


In [124]:
df=df_cleaned

# Удаление дубликатов

In [125]:
duplicates = df.duplicated(subset=['solve_time', 'pll','STPS'], keep=False)

df[duplicates].head()

,solve_id,solver,solve_time,pll,pll_time,Split,STM,STPS,ETM,ETPS


В датасете нет одинаковых объектов.

In [126]:
df = df[df.STM > df.STM.min()]

In [127]:
df = df[(df.pll_time > df.pll_time.mean()-3*df.pll_time.std()) & (df.pll_time < df.pll_time.mean()+3*df.pll_time.std())]

In [128]:
df.describe()

,solve_id,solve_time,pll_time,STM,STPS,ETM,ETPS
count,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000,698.000000
mean,11753.515759,4.993883,1.238553,14.723496,12.414542,14.906877,12.567665
std,692.834587,1.007150,0.318596,3.447419,3.466687,3.291428,3.367549
min,10523.000000,2.850000,0.630000,7.000000,3.930000,7.000000,4.210000
25%,11094.250000,4.280000,1.002500,12.000000,9.922500,13.000000,10.245000
50%,11700.500000,4.890000,1.200000,15.000000,12.085000,15.000000,12.215000
75%,12328.750000,5.547500,1.430000,17.000000,14.747500,17.000000,14.765000
max,12922.000000,10.960000,2.470000,27.000000,22.340000,27.000000,22.340000


Все выбросы и нулевые значения удалены

In [129]:
df.shape

(698, 10)

Осталось 698 объектов. При необходимости можно будет напарсить больше данных

# Нормализую время по каждому сборщику
Для этого создам столбец с количеством сборок каждого сборщика и удалим сборщиков, у которых меньше 8 сборок, так как про них недостаточно данных для нормализации их времени сборки.

In [130]:
counts = df.groupby("solver").ETM.count()

In [131]:
mean_df = df.groupby("solver").pll_time.mean()
std_df = df.groupby("solver").pll_time.std()

In [132]:
def fun(solver):
        return counts[solver]

df["Solver_count"] = df.solver.apply(fun)
df = df[df["Solver_count"]>7]

In [133]:
df.shape

(630, 11)

In [140]:
df["pll_time"] = df.apply(lambda row: round((row['pll_time'] - mean_df[row['solver']]) / std_df[row['solver']],6), axis=1)
df.head()

,solve_id,solver,solve_time,pll,pll_time,Split,STM,STPS,ETM,ETPS,...,F,F2,R,R2,B,B2,D,D2,simul,ghost_move
5,12921,Dylan Miller,4.40,U U' R2 U R' U R' U' R U' R2 D U' R' U R D' U2',-2.798777,31.1%,18,13.14,18,13.14,...,0,0,5,2,0,0,2,0,2,1
6,12920,Dylan Miller,4.60,U' R' U2 R' D' R U' R' D R U R U' R' U' R,-7.075866,22.4%,16,15.53,16,15.53,...,0,0,8,0,0,0,2,0,0,0
7,12919,Dylan Miller,5.26,U' R2 U R' U R' U' R U' R2 U' D R' U R D' U,-0.786030,29.1%,17,11.11,17,11.11,...,0,0,5,2,0,0,2,0,2,1
8,12918,Dylan Miller,6.72,U U' R' U' R U D' R2 U R' U R U' R U' R2' U' D U',4.497432,29.0%,19,9.74,19,9.74,...,0,0,5,2,0,0,2,0,3,0
9,12917,Dylan Miller,4.27,U R U R' F' R U R' U' R' F R2 U' R' U,-9.466003,19.7%,15,17.86,15,17.86,...,2,0,6,1,0,0,0,0,0,0


# Дополнительные фичи

In [135]:
for move in "ULFRBD":
  df[f"{move}"]=df["pll"].apply(lambda x: x.count(f"{move}")-x.count(f"{move}2"))
  df[f"{move}2"]=df["pll"].apply(lambda x: x.count(f"{move}2"))
df.head()

,solve_id,solver,solve_time,pll,pll_time,Split,STM,STPS,ETM,ETPS,...,L,L2,F,F2,R,R2,B,B2,D,D2
5,12921,Dylan Miller,4.40,U U' R2 U R' U R' U' R U' R2 D U' R' U R D' U2',0.453137,31.1%,18,13.14,18,13.14,...,0,0,0,0,5,2,0,0,2,0
6,12920,Dylan Miller,4.60,U' R' U2 R' D' R U' R' D R U R U' R' U' R,-0.752769,22.4%,16,15.53,16,15.53,...,0,0,0,0,8,0,0,0,2,0
7,12919,Dylan Miller,5.26,U' R2 U R' U R' U' R U' R2 U' D R' U R D' U,1.020622,29.1%,17,11.11,17,11.11,...,0,0,0,0,5,2,0,0,2,0
8,12918,Dylan Miller,6.72,U U' R' U' R U D' R2 U R' U R U' R U' R2' U' D U',2.510271,29.0%,19,9.74,19,9.74,...,0,0,0,0,5,2,0,0,2,0
9,12917,Dylan Miller,4.27,U R U R' F' R U R' U' R' F R2 U' R' U,-1.426658,19.7%,15,17.86,15,17.86,...,0,0,2,0,6,1,0,0,0,0


Создаю дополнительные фичи. Какие ходы и комбинации ходов присутствуют в решении.

In [136]:
df["simul"]=df["pll"].apply(lambda x: x.count("U D")+x.count("U' D")+x.count("U2 D")+x.count("D U")+x.count("D' U")+x.count("D2 U"))

In [137]:
df["ghost_move"]=df["pll"].apply(lambda x: x.count("R' U R'") + x.count("L U' L"))
df.head()

,solve_id,solver,solve_time,pll,pll_time,Split,STM,STPS,ETM,ETPS,...,F,F2,R,R2,B,B2,D,D2,simul,ghost_move
5,12921,Dylan Miller,4.40,U U' R2 U R' U R' U' R U' R2 D U' R' U R D' U2',0.453137,31.1%,18,13.14,18,13.14,...,0,0,5,2,0,0,2,0,2,1
6,12920,Dylan Miller,4.60,U' R' U2 R' D' R U' R' D R U R U' R' U' R,-0.752769,22.4%,16,15.53,16,15.53,...,0,0,8,0,0,0,2,0,0,0
7,12919,Dylan Miller,5.26,U' R2 U R' U R' U' R U' R2 U' D R' U R D' U,1.020622,29.1%,17,11.11,17,11.11,...,0,0,5,2,0,0,2,0,2,1
8,12918,Dylan Miller,6.72,U U' R' U' R U D' R2 U R' U R U' R U' R2' U' D U',2.510271,29.0%,19,9.74,19,9.74,...,0,0,5,2,0,0,2,0,3,0
9,12917,Dylan Miller,4.27,U R U R' F' R U R' U' R' F R2 U' R' U,-1.426658,19.7%,15,17.86,15,17.86,...,2,0,6,1,0,0,0,0,0,0


In [142]:
print(df.shape[0])
len(df.pll.unique())

630


465

In [139]:
df.to_csv('rubik_solves_cleaned.csv', index=False)

Получили выборку 711 объектов и 21 фичу(3 столбца не участвуют)
